In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/telco_churn.csv")

# Nettoyage (le même que dans l'EDA)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID"])

print(df.shape)

(7043, 20)


In [3]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5634, 19) (1409, 19)
0.2653532126375577 0.2654364797728886


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colonnes_num = ["tenure", "MonthlyCharges", "TotalCharges"]
colonnes_cat = [c for c in X.columns if c not in colonnes_num]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), colonnes_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), colonnes_cat),
])

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

modele_log = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

modele_log.fit(X_train, y_train)

y_pred = modele_log.predict(X_test)
y_proba = modele_log.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC :", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC : 0.841501976284585


## Modèle de référence : régression logistique
Premier modèle simple, qui sert de point de comparaison. Résultats sur le jeu de test sont: 
ROC-AUC = 0,84, recall = 0,78 et precision = 0,50 pour la classe « churn ». Le modèle détecte 78 % des clients qui partent, au prix de nombreuses fausses alertes (une alerte sur deux est correcte). 

In [8]:
from sklearn.ensemble import RandomForestClassifier

modele_rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        min_samples_leaf=5, random_state=42, n_jobs=-1)),
])

modele_rf.fit(X_train, y_train)
y_pred_rf = modele_rf.predict(X_test)
y_proba_rf = modele_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC :", roc_auc_score(y_test, y_proba_rf))

              precision    recall  f1-score   support

           0       0.90      0.76      0.82      1035
           1       0.54      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409

ROC-AUC : 0.8417112299465241


In [9]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()

modele_xgb = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        scale_pos_weight=ratio, eval_metric="logloss", random_state=42)),
])

modele_xgb.fit(X_train, y_train)
y_pred_xgb = modele_xgb.predict(X_test)
y_proba_xgb = modele_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC :", roc_auc_score(y_test, y_proba_xgb))

              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409

ROC-AUC : 0.8422963651863906


## Comparaison des modèles

Les trois modèles ont des performances équivalentes (ROC-AUC ≈ 0,84) car les écarts sont trop faibles pour être significatifs. Je retiens XGBoost pour son meilleur recall et sa compatibilité avec SHAP. Sur ce jeu de données, la complexité du modèle apporte peu où l'information utile est surtout dans les variables (contrat, ancienneté, charges).

In [10]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(modele_xgb, "../models/churn_model.joblib")

['../models/churn_model.joblib']

In [11]:
df_clean = pd.read_csv("../data/telco_churn.csv")
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce").fillna(0)
df_clean["Churn"] = (df_clean["Churn"] == "Yes").astype(int)
df_clean.to_csv("../data/telco_clean.csv", index=False)